In [ ]:
!pip install openai pandas tqdm # для запросов к моделям через OpenRouter

import os
import time
import json
import random
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from tqdm import tqdm
from openai import OpenAI
from getpass import getpass # для безопасного ввода API-ключа.

In [ ]:
# ввожу ключ от OpenRouter
os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

OpenRouter API key: ··········


In [ ]:
client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
#Загрузка подготовленного датасета (школьники + сгенерированные тексты)
df = pd.read_csv("final_dataset.csv")
df = df.dropna(subset=["text", "label"])
df["text"] = df["text"].astype(str)
df["label"] = df["label"].str.lower()
print(df["label"].value_counts())
df.head()

label
human    1236
ai       1212
Name: count, dtype: int64


,source_id,text,label,model,prompt_type
0,0,В своем произведении Горький ставит вопрос: «Ч...,human,human,none
1,1,"Каждый член семьи незаменим, ведь родные для н...",human,human,none
2,2,"Каждый человек должен не забывать читать, ведь...",human,human,none
3,3,"Пожалуй, ни для кого не будет секретом, что А....",human,human,none
4,4,Доброта — это хорошее отношение к другим. В да...,human,human,none


In [ ]:
# определяю модели для бейзлайна

BASELINE_MODELS = [
  #  "x-ai/grok-4.1-fast",
   # "openai/gpt-5.4-mini"
    "deepseek/deepseek-v4-flash"
]

In [ ]:
# Задаю промт для детекции


DETECTION_PROMPT = """
Ты эксперт по анализу школьных сочинений на русском языке.

Тебе дан текст сочинения.
Нужно определить:
- написан ли текст человеком;
- или сгенерирован языковой моделью.

Важно:
Современные LLM способны:
- имитировать школьный стиль;
- использовать разговорные конструкции;
- допускать мелкие ошибки;
- делать текст менее академичным;
- намеренно humanize output.

Поэтому:
- ошибки не означают human;
- разговорный стиль не обязательно  human;
- эмоциональность не обязательно означает human.

Однако:
- многие реальные школьные сочинения также:
  - неровные;
  - противоречивые;
  - шаблонные;
  - эмоциональные;
  - плохо структурированные.

Поэтому нельзя автоматически считать текст AI-generated только из-за:
- хорошей связности;
- литературности;
- шаблонных рассуждений;
- плавных переходов.

Анализируй более глубокие признаки:
- статистическую сглаженность текста;
- повторяемость структуры аргументации;
- чрезмерно универсальные выводы;
- одинаковый ритм абзацев;
- отсутствие естественных семантических неровностей;
- неестественно стабильный тон сочинения;
- искусственную равномерность качества текста.

Особенно важно:
Реальные школьные тексты часто содержат:
- случайные скачки мысли;
- неровную логику;
- локальные противоречия;
- неидеальные переходы;
- избыточные или странные фразы;
- эмоциональные перегибы.

Не классифицируй текст как AI без достаточного количества признаков генерации.

Если признаки генерации неоднозначны,
оцени совокупность stylistic и semantic паттернов.
Не используй все время слишком консервативную или слишком агрессивную стратегию.

Сначала  проанализируй весь текст по этим правилам,
В ответе верни только JSON.

Формат ответа:

{{
  "label": "human" или "ai",
  "confidence": число от 0 до 1
}}

Никаких пояснений.
Никакого markdown.
Только JSON.

Текст:
{text}
"""


In [ ]:
# Функция для детекции
# на выходе по каждому тиексту получаю pred_label - human или ai и  confidence - уверенность модели

def detect_text(model, text, max_retries=3):

    prompt = DETECTION_PROMPT.format(
        text=text
    )

    for attempt in range(max_retries):

        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "system",
                        "content": "Ты AI detector. "
                            "Отвечай только валидным JSON. "
                            "Без пояснений."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0,
                max_tokens=100
            )

            raw_answer = response.choices[0].message.content.strip()
            data = json.loads(raw_answer)
            pred_label = data["label"].lower()
            confidence = float(data["confidence"])
            return pred_label, confidence, raw_answer

        except Exception as e:
            print("Ошибка:", e)
            print("Попытка:", attempt + 1)

            time.sleep(3)

    return None, None, None

In [ ]:
# прогоняем все тексты через все baseline-модели

results_path = "baseline_results.csv"

results = []

for model in BASELINE_MODELS:

    print(f"\n baseline model: {model}")

    for i, row in tqdm(df.iterrows(), total=len(df)):

        text = row["text"]
        true_label = row["label"]

        pred_label, confidence, raw_answer = detect_text(
            model=model,
            text=text
        )

        result = {
            "id": i,
            "text": text,
            "true_label": true_label,
            "pred_label": pred_label,
            "confidence": confidence,
            "baseline_model": model,
            "raw_answer": raw_answer
        }

        results.append(result)

        #  результат
        pd.DataFrame(results).to_csv(
            results_path,
            index=False,
            encoding="utf-8-sig"
        )

        time.sleep(1)

baseline_df = pd.DataFrame(results)

baseline_df.head()


 baseline model: deepseek/deepseek-v4-flash


  0%|          | 0/2448 [00:00<?, ?it/s]

Ошибка: 'NoneType' object has no attribute 'strip'
Попытка: 1


  0%|          | 2/2448 [00:11<3:34:59,  5.27s/it]

Ошибка: 'NoneType' object has no attribute 'strip'
Попытка: 1


...

Ошибка: 'NoneType' object has no attribute 'strip'
Попытка: 1
Ошибка: 'NoneType' object has no attribute 'strip'
Попытка: 2
Ошибка: 'NoneType' object has no attribute 'strip'
Попытка: 3


 97%|█████████▋| 2383/2448 [4:32:51<12:49, 11.84s/it]

Ошибка: 'NoneType' object has no attribute 'strip'
Попытка: 1
Ошибка: 'NoneType' object has no attribute 'strip'
Попытка: 2


100%|██████████| 2448/2448 [4:36:35<00:00,  6.78s/it]


,id,text,true_label,pred_label,confidence,baseline_model,raw_answer
0,0,В своем произведении Горький ставит вопрос: «Ч...,human,human,0.85,deepseek/deepseek-v4-flash,"{\n ""label"": ""human"",\n ""confidence"": 0.85\n}"
1,1,"Каждый член семьи незаменим, ведь родные для н...",human,ai,0.85,deepseek/deepseek-v4-flash,"{\n ""label"": ""ai"",\n ""confidence"": 0.85\n}"
2,2,"Каждый человек должен не забывать читать, ведь...",human,human,0.85,deepseek/deepseek-v4-flash,"{\n ""label"": ""human"",\n ""confidence"": 0.85\n}"
3,3,"Пожалуй, ни для кого не будет секретом, что А....",human,human,0.85,deepseek/deepseek-v4-flash,"{\n ""label"": ""human"",\n ""confidence"": 0.85\n}"
4,4,Доброта — это хорошее отношение к другим. В да...,human,human,0.85,deepseek/deepseek-v4-flash,"{\n ""label"": ""human"",\n ""confidence"": 0.85\n}"


In [ ]:
baseline_df[
    baseline_df["baseline_model"] ==  "deepseek/deepseek-v4-flash"
]["pred_label"].value_counts(dropna=False)

,count
pred_label,
human,2046
ai,233
None,169


In [ ]:
baseline_df["true_label"].value_counts()

,count
true_label,
human,1236
ai,1212


In [ ]:
baseline_df[[
    "baseline_model",
    "true_label",
    "pred_label",
    "confidence",
    "raw_answer"
]].head(150)

,baseline_model,true_label,pred_label,confidence,raw_answer
0,deepseek/deepseek-v4-flash,human,human,0.85,"{\n ""label"": ""human"",\n ""confidence"": 0.85\n}"
1,deepseek/deepseek-v4-flash,human,ai,0.85,"{\n ""label"": ""ai"",\n ""confidence"": 0.85\n}"
2,deepseek/deepseek-v4-flash,human,human,0.85,"{\n ""label"": ""human"",\n ""confidence"": 0.85\n}"
3,deepseek/deepseek-v4-flash,human,human,0.85,"{\n ""label"": ""human"",\n ""confidence"": 0.85\n}"
4,deepseek/deepseek-v4-flash,human,human,0.85,"{\n ""label"": ""human"",\n ""confidence"": 0.85\n}"
...,...,...,...,...,...
145,deepseek/deepseek-v4-flash,human,human,0.85,"{\n ""label"": ""human"",\n ""confidence"": 0.85\n}"
146,deepseek/deepseek-v4-flash,human,human,0.85,"{\n ""label"": ""human"",\n ""confidence"": 0.85\n}"
147,deepseek/deepseek-v4-flash,human,human,0.85,"{\n ""label"": ""human"",\n ""confidence"": 0.85\n}"
148,deepseek/deepseek-v4-flash,human,human,0.70,"{\n ""label"": ""human"",\n ""confidence"": 0.7\n}"


In [ ]:
# метрики  отдельно для каждой baseline-модели

metrics = []

for model in BASELINE_MODELS:

    part = baseline_df[
        baseline_df["baseline_model"] == model
    ].dropna(subset=["pred_label"])

    y_true = part["true_label"].map({
        "human": 0,
        "ai": 1
    })

    y_pred = part["pred_label"].map({
        "human": 0,
        "ai": 1
    })

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    metrics.append({
        "model": model,
        "accuracy": acc,
        "precision_ai": prec,
        "recall_ai": rec,
        "f1_ai": f1,
        "n": len(part)
    })

metrics_df = pd.DataFrame(metrics)

metrics_df

,model,accuracy,precision_ai,recall_ai,f1_ai,n
0,deepseek/deepseek-v4-flash,0.463361,0.180258,0.039106,0.064269,2279


In [ ]:
# по каждой модели confusion matrix

for model in BASELINE_MODELS:
    print(model)
    part = baseline_df[
        baseline_df["baseline_model"] == model
    ].dropna(subset=["pred_label"])

    y_true = part["true_label"].map({
        "human": 0,
        "ai": 1
    })

    y_pred = part["pred_label"].map({
        "human": 0,
        "ai": 1
    })

    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["human", "ai"]
        )
    )

deepseek/deepseek-v4-flash
Confusion matrix:
[[1014  191]
 [1032   42]]

Classification report:
              precision    recall  f1-score   support

       human       0.50      0.84      0.62      1205
          ai       0.18      0.04      0.06      1074

    accuracy                           0.46      2279
   macro avg       0.34      0.44      0.34      2279
weighted avg       0.35      0.46      0.36      2279



In [ ]:
# сохраню результаты

baseline_df.to_csv(
    "llm_baseline_results.csv",
    index=False,
    encoding="utf-8-sig"
)

metrics_df.to_csv(
    "llm_baseline_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

from google.colab import files

files.download("llm_baseline_results.csv")
files.download("llm_baseline_metrics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Попробую проверииь качество бейзлайна посмоотреа roc_auk и разные трешхолды

In [14]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# список файлов с результатами бейзлайнов
result_files = [
    "llm_baseline_results_ Gemini 3 Flash Preview.csv",
    "llm_baseline_results_deepseek-v4-flash.csv",
    "llm_baseline_results_openaigpt-5_4-mini.csv"
]

all_metrics = []

for file_name in result_files:

    df = pd.read_csv(file_name)
    df["confidence"] = pd.to_numeric(df["confidence"], errors="coerce")
        # убираю плохие строки
    df = df.dropna(subset=["true_label", "pred_label", "confidence"]).copy()
    df["y_true"] = (df["true_label"] == "ai").astype(int)

    # делаю score для класса ai
    df["ai_score"] = df.apply(
        lambda row: row["confidence"] if row["pred_label"] == "ai" else 1 - row["confidence"],
        axis=1
    )


    # считаю auc
    roc_auc = roc_auc_score(df["y_true"], df["ai_score"])
    pr_auc = average_precision_score(df["y_true"], df["ai_score"])

    # подбираю лучший threshold
    thresholds = np.arange(0.05, 1.00, 0.05)

    best_f1 = -1
    best_threshold = None
    best_precision = None
    best_recall = None
    best_accuracy = None

    for thr in thresholds:
        y_pred = (df["ai_score"] >= thr).astype(int)

        f1 = f1_score(df["y_true"], y_pred, zero_division=0)
        precision = precision_score(df["y_true"], y_pred, zero_division=0)
        recall = recall_score(df["y_true"], y_pred, zero_division=0)
        accuracy = accuracy_score(df["y_true"], y_pred)

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thr
            best_precision = precision
            best_recall = recall
            best_accuracy = accuracy

    all_metrics.append({
        "file_name": file_name,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "best_threshold": best_threshold,
        "best_f1_ai": best_f1,
        "precision_ai": best_precision,
        "recall_ai": best_recall,
        "accuracy": best_accuracy,
        "n_rows": len(df)
    })

final_metrics_df = pd.DataFrame(all_metrics).sort_values("roc_auc", ascending=False)

final_metrics_df

,file_name,roc_auc,pr_auc,best_threshold,best_f1_ai,precision_ai,recall_ai,accuracy,n_rows
1,llm_baseline_results_deepseek-v4-flash.csv,0.367650,0.418643,0.05,0.640620,0.471259,1.0,0.471259,2279
0,llm_baseline_results_ Gemini 3 Flash Preview.csv,0.326000,0.424683,0.05,0.666667,0.500000,1.0,0.500000,100
2,llm_baseline_results_openaigpt-5_4-mini.csv,0.214186,0.354625,0.05,0.662295,0.495098,1.0,0.495098,2448
